# 02 -- Temporal Workflows

This notebook covers in-memory temporal cubes, file-backed GeoTIFF series,
incremental writing with progress, streaming reductions, and a screening
workflow.  It draws from command-line examples 05 through 10.

**Run the individual scripts:** `05_temporal_cube.py`,
`06_file_backed_series.py`, `07_incremental_writer.py`,
`08_streaming_reductions.py`, `10_landing_site_screening.py`

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


import lunarscout as ls
import numpy as np

from _example_support import (
    ensure_synthetic_scenario,
    synthetic_georef,
    synthetic_dem,
    synthetic_times,
    synthetic_temporal_cube,
    ensure_synthetic_series,
)

WORKSPACE = Path("/tmp/lunarscout_notebook_02")
WORKSPACE.mkdir(parents=True, exist_ok=True)

ensure_synthetic_scenario(WORKSPACE)
series = ensure_synthetic_series(WORKSPACE)
georef = synthetic_georef()
cube = synthetic_temporal_cube(georef)

SCENARIO = ls.open_scenario(WORKSPACE / "synthetic_scenario")

print(f"Workspace: {WORKSPACE}")

---
## 1. In-Memory Temporal Cube

A `TemporalCube` holds a complete `(time, y, x)` NumPy array with UTC time
coordinates.  Use it for modest arrays and exploratory work.

In [ ]:
print(f"  shape:      {cube.values.shape}")
print(f"  dtype:      {cube.values.dtype}")
print(f"  time start: {cube.times[0]}")
print(f"  time end:   {cube.times[-1]}")
print(f"  time steps: {len(cube.times)}")
print(f"  memory:     {cube.values.nbytes / 1024:.1f} KiB")

In [ ]:
# Reduce the time axis into per-pixel statistics.
mean, mean_georef = ls.temporal_mean(cube)
_min, _ = ls.temporal_min(cube)
_max, _ = ls.temporal_max(cube)
std, _ = ls.temporal_std(cube)

print(f"Per-pixel mean:   [{mean.min():.3f}, {mean.max():.3f}]")
print(f"Per-pixel std:    [{std.min():.3f}, {std.max():.3f}]")

In [ ]:
_temporal = SCENARIO.output_path("analysis/temporal")
_temporal.mkdir(parents=True, exist_ok=True)
for name, arr in [("mean", mean), ("minimum", _min), ("maximum", _max), ("standard_deviation", std)]:
    ls.write_geotiff(str(_temporal / f"{name}.tif"), arr, mean_georef, overwrite=True)
print(f"Temporal reducers written under {_temporal}")

**Try this:** create a `TemporalCube` from a NumPy array of your own
choosing, then try the same four reducers.

---
## 2. File-Backed Temporal Series

Store large time series as one single-band GeoTIFF per timestamp, plus a
manifest and optional VRT.  The series has no `.values` property -- this
prevents accidentally materialising a large cube.

In [ ]:
print(f"Series root:   {series.root}")
print(f"Signal name:   {series.signal_name}")
print(f"VRT path:      {series.vrt_path}")

# Read by zero-based layer index.
layer0, _ = series.read_layer(0)
print(f"\nLayer 0 shape: {layer0.shape}, min={layer0.min():.3f}, max={layer0.max():.3f}")

In [ ]:
# Read nearest to a specific UTC time.
from datetime import datetime, timezone
target = datetime(2027, 1, 1, 2, 30, tzinfo=timezone.utc)
nearest, _ = series.read_time(target, method="nearest")
print(f"Nearest to {target}: shape={nearest.shape}")

**Try this:** read the layer for a specific index and print its raster
statistics using NumPy.

---
## 3. Incremental Writer

Write directly through `TemporalGeoTiffSeriesWriter` without first building
a `TemporalCube`.  Memory stays proportional to one layer.

In [ ]:
from datetime import datetime, timedelta, timezone

output_root = SCENARIO.output_path("analysis/incremental_sun")
print(f"Writing to {output_root}")

def _on_progress(progress):
    print(f"  wrote {progress.last_time}")

with ls.TemporalGeoTiffSeriesWriter(
    output_root,
    georef=georef,
    dtype=np.float32,
    signal_name="sun_fraction",
    units="fraction",
    progress_callback=_on_progress,
    overwrite=True,
) as w:
    start = datetime(2027, 1, 1, 0, 0, tzinfo=timezone.utc)
    for i in range(6):
        t = start + timedelta(hours=i)
        layer = np.sin(
            np.linspace(0, np.pi, georef.width * georef.height,
                        dtype=np.float32)
        ).reshape(georef.height, georef.width) * 0.5 + 0.5
        w.write_layer(t, layer)

incremental = w.result
print(f"\nFinalised: {incremental.root}")
incremental.close()

**Try this:** replace the sine-wave layer with a function that depends on
the pixel column index.

---
## 4. Streaming Temporal Reductions

The same reducer names (`temporal_mean`, `temporal_min`, ...) accept both
`TemporalCube` and `TemporalGeoTiffSeries`.  For a series they stream
layers without loading the full cube.

In [ ]:
s_mean, s_georef = ls.temporal_mean(series)
s_stdev, _ = ls.temporal_std(series)

print(f"Streamed mean range:   [{s_mean.min():.4f}, {s_mean.max():.4f}]")
print(f"Streamed std range:    [{s_stdev.min():.4f}, {s_stdev.max():.4f}]")

Compare with the in-memory reductions from section 1.  The results should
agree within floating-point rounding.

---
## 5. Landing-Site Screening

Combine slope and temporal mean illumination into a candidate mask, then
filter for regions of adequate size.  This mirrors `10_landing_site_screening.py`.

In [ ]:
dem, dem_georef = ls.read_geotiff(SCENARIO.dem_path())
slope_val, slope_georef = ls.slope(dem, dem_georef, output_nodata=-9999.0)
illum, illum_georef = ls.temporal_mean(series)

# Grid compatibility check before combining.
ls.require_same_grid(slope_georef, illum_georef)

# Illustrative thresholds -- not landing-site recommendations.
SLOPE_MAX = 8.0          # degrees
ILLUM_MIN = 0.60         # fraction
REGION_MIN = 80          # pixels

candidate = (slope_val <= SLOPE_MAX) & (illum >= ILLUM_MIN)
print(f"Candidate cells before filtering:  {candidate.sum()}")

large, large_georef = ls.filter_regions_by_size(
    candidate, slope_georef,
    threshold=REGION_MIN, comparator=">=",
)
borders, _ = ls.find_borders(large, large_georef)
print(f"Candidate cells after size filter: {large.sum()}")

In [ ]:
_screening = SCENARIO.output_path("analysis/screening")
_screening.mkdir(parents=True, exist_ok=True)
nodata_gref = large_georef.with_nodata(0)
ls.write_geotiff(str(_screening / "candidate_sites.tif"), large.astype("uint8"), nodata_gref, overwrite=True)
ls.write_geotiff(str(_screening / "candidate_borders.tif"), borders.astype("uint8"), nodata_gref, overwrite=True)
print(f"Screening outputs written under {_screening}")

**Try this:** vary `SLOPE_MAX`, `ILLUM_MIN`, and `REGION_MIN`.  How do the
candidate regions respond?  What happens when you set `ILLUM_MIN` to 1.0?

---
## Cleanup

In [ ]:
series.close()
print("Series closed.")